In [ ]:
import ctypes
ctypes.CDLL('/usr/lib/aarch64-linux-gnu/libgomp.so.1', mode=ctypes.RTLD_GLOBAL)

import os
import sys
sys.path.append('/usr/local/lib')
sys.path.append('/usr/local/lib/python3.6/pyrealsense2')

import pyrealsense2 as rs

import gc
import time
import math
import cv2
import numpy as np
import traitlets
import pyrealsense2 as rs
import ipywidgets as widgets
from IPython.display import display

from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from jetracer.nvidia_racecar import NvidiaRacecar


controller = widgets.Controller(index=0)
display(controller)

In [ ]:
car = NvidiaRacecar()
car.throttle_gain = 0.5
car.steering_offset = 0
car.steering_gain = -0.65
car.steering = 0
car.throttle = -0.25

steering_link = traitlets.dlink(
    (controller.axes[0], 'value'), 
    (car, 'steering'), 
    transform=lambda x: -x
)

throttle_link = traitlets.dlink(
    (controller.buttons[7], 'value'), 
    (car, 'throttle'), 
    transform=lambda x: -x
)

brake_link = traitlets.dlink(
    (controller.buttons[6], 'value'), 
    (car, 'throttle'), 
    transform=lambda x: x
)

In [ ]:
if 'camera' in globals():
    try:
        camera.running = False
        camera.unobserve_all()
        time.sleep(0.5)
        del camera
        gc.collect()
    except Exception:
        pass


camera = CSICamera(width=224, height=224, capture_fps=30, capture_device=0)


camera_widget = widgets.Image(format='jpeg', width=224, height=224)
map_widget = widgets.Image(format='jpeg', width=300, height=300)

def update_camera_image(change):
    camera_widget.value = bgr8_to_jpeg(change['new'])

camera.observe(update_camera_image, names='value')
camera.running = True

# Wyświetlamy kontroler, kamerę oraz mapę T265 w jednym wierszu
display(controller)
display(widgets.HBox([camera_widget, map_widget]))


def quaternion_yaw(qx, qy, qz, qw):
    siny_cosp = 2 * (qw * qy + qz * qx)
    cosy_cosp = 1 - 2 * (qx * qx + qy * qy)
    return math.atan2(siny_cosp, cosy_cosp)

pipe = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.pose)

print("Uruchamianie T265...")
pipe.start(config)

map_size = 300  # Dostosowane do rozmiaru widżetu
trail_map = np.zeros((map_size, map_size, 3), dtype=np.uint8)
traj = []
scale = 50      # Skalowanie metry na piksele
cent_x, cent_y = map_size // 2, map_size // 2

print("System gotowy! Pętla T265 wystartowała.")


In [ ]:

#main loop
try:
    while True:
        frames = pipe.poll_for_frames()
        if not frames:
            time.sleep(0.01)
            continue
            
        pose_frame = frames.get_pose_frame()
        if not pose_frame:
            continue

        data = pose_frame.get_pose_data()
        x_m = data.translation.x
        z_m = -data.translation.z
        y_m = data.translation.y
        
        pixel_x = int(cent_x + (x_m * scale))
        pixel_y = int(cent_y + (z_m * scale))
        current_pos = (pixel_x, pixel_y)
        
        if not traj or traj[-1] != current_pos:
            traj.append(current_pos)

        display_map = trail_map.copy()
        
        # Siatka
        for i in range(-5, 6):
            cv2.line(display_map, (cent_x + i * scale, 0), (cent_x + i * scale, map_size), (40, 40, 40), 1)
            cv2.line(display_map, (0, cent_y + i * scale), (map_size, cent_y + i * scale), (40, 40, 40), 1)

        # Ścieżka
        if len(traj) > 1:
            for i in range(1, len(traj)):
                cv2.line(display_map, traj[i-1], traj[i], (0, 255, 0), 2)

        # Kąt i pozycja autka
        qx, qy, qz, qw = data.rotation.x, data.rotation.y, data.rotation.z, data.rotation.w
        yaw = quaternion_yaw(qx, qy, qz, qw)
        arrow_length = 20
        arrow_end_x = int(pixel_x + arrow_length * math.sin(yaw))
        arrow_end_y = int(pixel_y + arrow_length * math.cos(yaw))
        
        cv2.arrowedLine(display_map, (pixel_x, pixel_y), (arrow_end_x, arrow_end_y), (0, 0, 255), 2, tipLength=0.2)
        cv2.circle(display_map, (pixel_x, pixel_y), 5, (255, 0, 0), -1)
        
        # Tekst z pozycją
        cv2.putText(display_map, f"X: {x_m:+.2f}m Z: {z_m:+.2f}m", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
        cv2.putText(display_map, f"YAW: {math.degrees(yaw):+.1f}deg", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

        # Bezpośrednia aktualizacja widżetu bez przerywania widoku
        _, encoded_img = cv2.imencode('.jpg', display_map)
        map_widget.value = encoded_img.tobytes()
        
        time.sleep(0.01)

except KeyboardInterrupt:
    print("\nZatrzymano przez użytkownika.")
except Exception as e:
    print(f"\nBłąd: {e}")
finally:
    pipe.stop()
    car.throttle = 0.0
    car.steering = 0.0
    camera.running = False
    print("Urządzenia zatrzymane pomyślnie.")